# LoRA Fine-tuning

В этом ноутбуке выполняется LoRA fine-tuning модели `Qwen/Qwen2.5-0.5B-Instruct` на датасете DialogSum.

Цель — адаптировать маленькую decoder-only модель к задаче dialogue summarization и сравнить качество с zero-shot baseline.

- Unsloth — ускоряет и облегчает fine-tuning на Colab GPU
- TRL/SFTTrainer — стандартный инструмент Hugging Face для supervised fine-tuning

## Imports

In [1]:
!nvidia-smi

Sun Jun  7 13:03:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U uv
!uv pip install --system unsloth --torch-backend=auto
!pip install -q kagglehub pandas numpy datasets trl wandb evaluate rouge_score sacrebleu

Using Python 3.12.13 environment at: /usr
Checked 1 package in 118ms


In [3]:
import os
import pandas as pd
import numpy as np

import kagglehub
from datasets import Dataset

## Uploading data

In [4]:
dataset_path = kagglehub.dataset_download("marawanxmamdouh/dialogsum")
print("Dataset path:", dataset_path)

Using Colab cache for faster access to the 'dialogsum' dataset.
Dataset path: /kaggle/input/dialogsum


In [5]:
csv_files = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

csv_files

['/kaggle/input/dialogsum/CSV/hiddentest_dialogue.csv',
 '/kaggle/input/dialogsum/CSV/hiddentest_topic.csv',
 '/kaggle/input/dialogsum/CSV/validation.csv',
 '/kaggle/input/dialogsum/CSV/train.csv',
 '/kaggle/input/dialogsum/CSV/test.csv']

In [6]:
dataframes = {}

for file_path in csv_files:
    file_name = os.path.basename(file_path)
    df_name = file_name.replace(".csv", "")
    dataframes[df_name] = pd.read_csv(file_path)

    print(df_name, dataframes[df_name].shape)

hiddentest_dialogue (100, 2)
hiddentest_topic (100, 2)
validation (500, 4)
train (12460, 4)
test (1500, 4)


In [7]:
train_df = dataframes["train"]
val_df = dataframes["validation"]
test_df = dataframes["test"]

train_df.head()

,id,dialogue,summary,topic
0,train_0,"#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. ...","Mr. Smith's getting a check-up, and Doctor Haw...",get a check-up
1,train_1,"#Person1#: Hello Mrs. Parker, how have you bee...",Mrs Parker takes Ricky for his vaccines. Dr. P...,vaccines
2,train_2,"#Person1#: Excuse me, did you see a set of key...",#Person1#'s looking for a set of keys and asks...,find keys
3,train_3,#Person1#: Why didn't you tell me you had a gi...,#Person1#'s angry because #Person2# didn't tel...,have a girlfriend
4,train_4,"#Person1#: Watsup, ladies! Y'll looking'fine t...",Malik invites Nikki to dance. Nikki agrees if ...,dance


## Setting



In [8]:
TRAIN_SIZE = 1000
VAL_SIZE = 100

train_small_df = train_df.head(TRAIN_SIZE).copy()
val_small_df = val_df.head(VAL_SIZE).copy()

train_small_df.shape, val_small_df.shape

((1000, 4), (100, 4))

In [9]:
def format_training_example(row):
    return f"""You are a helpful assistant. Summarize the following dialogue in one concise paragraph.

Dialogue:
{row["dialogue"]}

Summary:
{row["summary"]}"""

In [10]:
train_small_df["text"] = train_small_df.apply(format_training_example, axis=1)
val_small_df["text"] = val_small_df.apply(format_training_example, axis=1)

train_small_df[["text"]].head(1).iloc[0]["text"]

"You are a helpful assistant. Summarize the following dialogue in one concise paragraph.\n\nDialogue:\n#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information bef

In [11]:
train_dataset = Dataset.from_pandas(train_small_df[["text"]])
val_dataset = Dataset.from_pandas(val_small_df[["text"]])

train_dataset, val_dataset

(Dataset({
     features: ['text'],
     num_rows: 1000
 }),
 Dataset({
     features: ['text'],
     num_rows: 100
 }))

## Unsloth

In [12]:
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


ERROR:bitsandbytes.cextension:bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No such file or directory


🦥 Unsloth Zoo will now patch everything to make training faster!


In [13]:
MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct"

max_seq_length = 2048
dtype = None
load_in_4bit = False

In [14]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-0.5B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Во время LoRA fine-tuning обучение успешно запустилось: training loss начал снижаться с 2.12 до 1.81 на первых шагах. Однако запуск завершился ошибкой, когда Trainer попытался сохранить промежуточный checkpoint на шаге 20.

Ошибка была связана с сериализацией checkpoint: `PicklingError: Can't pickle SFTConfig`. Эта проблема не относится к самому процессу обучения модели, а связана с автоматическим сохранением промежуточных checkpoint-ов.

Чтобы избежать этой ошибки, промежуточное сохранение checkpoint-ов было отключено с помощью параметра `save_strategy="no"`. После завершения обучения LoRA adapter сохраняется вручную через `model.save_pretrained(...)`.

In [15]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth 2026.6.1 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


In [19]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        output_dir="outputs/checkpoints/qwen_lora_dialogsum",

        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        warmup_steps=10,
        max_steps=60,
        learning_rate=2e-4,

        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        logging_steps=5,

        save_strategy="no",

        optim="adamw_torch",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

In [20]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
5,1.695271
10,1.721415
15,1.713619
20,1.723083
25,1.799470
30,1.754502
35,1.765254
40,1.730378
45,1.727395
50,1.585487


In [21]:
adapter_path = "outputs/checkpoints/qwen_lora_dialogsum_adapter"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("Saved adapter to:", adapter_path)

Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoints/qwen_lora_dialogsum_adapter/tokenizer_config.json.


Saved adapter to: outputs/checkpoints/qwen_lora_dialogsum_adapter


In [24]:
!zip -r qwen_lora_dialogsum_adapter.zip outputs/checkpoints/qwen_lora_dialogsum_adapter

  adding: outputs/checkpoints/qwen_lora_dialogsum_adapter/ (stored 0%)
  adding: outputs/checkpoints/qwen_lora_dialogsum_adapter/tokenizer.json (deflated 81%)
  adding: outputs/checkpoints/qwen_lora_dialogsum_adapter/tokenizer_config.json (deflated 89%)
  adding: outputs/checkpoints/qwen_lora_dialogsum_adapter/adapter_model.safetensors (deflated 7%)
  adding: outputs/checkpoints/qwen_lora_dialogsum_adapter/README.md (deflated 65%)
  adding: outputs/checkpoints/qwen_lora_dialogsum_adapter/chat_template.jinja (deflated 71%)
  adding: outputs/checkpoints/qwen_lora_dialogsum_adapter/adapter_config.json (deflated 59%)


In [25]:
from google.colab import files

files.download("qwen_lora_dialogsum_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Saving LoRA adapter

После завершения обучения был сохранён LoRA adapter.

LoRA adapter — это небольшой набор обучаемых весов, который хранит изменения, полученные во время fine-tuning. Базовая модель при этом не сохраняется целиком.

Для повторного использования fine-tuned модели нужно загрузить базовую модель `Qwen/Qwen2.5-0.5B-Instruct` и применить к ней сохранённый LoRA adapter.

Adapter был сохранён отдельно, чтобы не запускать обучение заново и использовать результат fine-tuning для последующего inference и оценки качества.

## LoRA fine-tuning result

LoRA fine-tuning был успешно запущен на подмножестве из 1000 train-примеров DialogSum.

Обучалась не вся модель, а только LoRA-адаптеры: 8,798,208 trainable parameters из 502,830,976 параметров модели, то есть около 1.75%.

Обучение было ограничено 60 шагами. В логах видно, что training loss изменялся в процессе обучения, а сам training loop успешно дошёл до конца. Это подтверждает, что LoRA fine-tuning был корректно запущен.

После завершения обучения LoRA adapter был сохранён отдельно через `model.save_pretrained(...)`.

## Inference after LoRA fine-tuning

In [26]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896, padding_idx=151665)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
    

In [27]:
EVAL_SIZE = 100

eval_df = test_df.head(EVAL_SIZE).copy()
eval_df.shape

(100, 4)

In [28]:
def build_prompt(dialogue):
    return f"""You are a helpful assistant. Summarize the following dialogue in one concise paragraph.

Dialogue:
{dialogue}

Summary:"""

In [31]:
import torch

def generate_lora_summary(dialogue, max_new_tokens=80):
    prompt = build_prompt(dialogue)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Отрезаем prompt, оставляем только сгенерированное summary
    summary = generated_text.split("Summary:")[-1].strip()

    return summary

In [32]:
example_dialogue = eval_df.iloc[0]["dialogue"]
reference_summary = eval_df.iloc[0]["summary"]

lora_summary = generate_lora_summary(example_dialogue)

print("REFERENCE SUMMARY:")
print(reference_summary)

print("\nLORA SUMMARY:")
print(lora_summary)

Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

REFERENCE SUMMARY:
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

LORA SUMMARY:
#Person1# asks Ms. Dawson to take a dictation for her. Ms. Dawson tells #Person1# that she needs to take a dictation for her and that it should go out as an intra-office memorandum to all employees by this afternoon. #Person1# asks Ms. Dawson if it applies to intra-office communications only or to all communications. Ms. Dawson says it applies to


In [33]:
from tqdm import tqdm

lora_summaries = []

for dialogue in tqdm(eval_df["dialogue"].tolist()):
    summary = generate_lora_summary(dialogue)
    lora_summaries.append(summary)

eval_df["qwen_lora_summary"] = lora_summaries

 75%|███████▌  | 75/100 [04:25<01:28,  3.54s/it]Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_ME

In [34]:
for i in range(5):
    row = eval_df.iloc[i]

    print("=" * 100)
    print("TOPIC:", row["topic"])

    print("\nREFERENCE SUMMARY:")
    print(row["summary"])

    print("\nQWEN LORA SUMMARY:")
    print(row["qwen_lora_summary"])

TOPIC: communication method

REFERENCE SUMMARY:
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

QWEN LORA SUMMARY:
#Person1# asks Ms. Dawson to take a dictation for her. Ms. Dawson tells #Person1# that she needs to take a dictation for her and that it should go out as an intra-office memorandum to all employees by this afternoon. #Person1# asks Ms. Dawson if it applies to intra-office communications only or to all communications. Ms. Dawson says it applies to
TOPIC: company policy

REFERENCE SUMMARY:
In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use of those programs and asks Ms. Dawson to send out a memo to all employees by the afternoon.

QWEN LORA SUMMARY:
#Person1# asks Ms. Dawson to take a dictation for her. Ms. Dawson tells #Person1# that she needs to take a dictation for her and that it should go ou

In [35]:
import evaluate

rouge_metric = evaluate.load("rouge")
sacrebleu_metric = evaluate.load("sacrebleu")

In [36]:
predictions = eval_df["qwen_lora_summary"].tolist()
references = eval_df["summary"].tolist()

rouge_results = rouge_metric.compute(
    predictions=predictions,
    references=references
)

bleu_results = sacrebleu_metric.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

qwen_lora_metrics = {
    "method": "qwen2.5_0.5b_instruct_lora",
    "eval_size": len(eval_df),
    "bleu": bleu_results["score"],
    "rouge1": rouge_results["rouge1"],
    "rouge2": rouge_results["rouge2"],
    "rougeL": rouge_results["rougeL"],
    "rougeLsum": rouge_results["rougeLsum"],
}

qwen_lora_metrics

{'method': 'qwen2.5_0.5b_instruct_lora',
 'eval_size': 100,
 'bleu': 6.613557180584074,
 'rouge1': np.float64(0.2595949891551792),
 'rouge2': np.float64(0.07162151625603753),
 'rougeL': np.float64(0.19571756071393526),
 'rougeLsum': np.float64(0.19570945364218853)}

## LoRA evaluation result

После LoRA fine-tuning модель `Qwen/Qwen2.5-0.5B-Instruct` была оценена на том же подмножестве test set из 100 примеров.

LoRA-модель показала улучшение по сравнению с zero-shot baseline по всем основным метрикам: BLEU, ROUGE-1, ROUGE-2 и ROUGE-L.

Это означает, что даже короткое дообучение на 1000 train-примерах и 60 training steps позволило модели лучше адаптироваться к задаче dialogue summarization и стилю эталонных summary в DialogSum.

При этом абсолютные значения метрик остаются умеренными, так как BLEU и ROUGE оценивают лексическое пересечение с reference summary и не всегда отражают смысловое качество generated summary.

In [37]:
baseline_eval_metrics = {
    "method": "first_last_sentence_baseline",
    "eval_size": 100,
    "bleu": 4.349586,
    "rouge1": 0.188139,
    "rouge2": 0.020102,
    "rougeL": 0.152581,
    "rougeLsum": 0.152428,
}

qwen_zero_shot_metrics = {
    "method": "qwen2.5_0.5b_instruct_zero_shot_vllm",
    "eval_size": 100,
    "bleu": 5.384482,
    "rouge1": 0.224649,
    "rouge2": 0.064147,
    "rougeL": 0.175893,
    "rougeLsum": 0.175831,
}

final_comparison_df = pd.DataFrame([
    baseline_eval_metrics,
    qwen_zero_shot_metrics,
    qwen_lora_metrics
])

final_comparison_df

,method,eval_size,bleu,rouge1,rouge2,rougeL,rougeLsum
0,first_last_sentence_baseline,100,4.349586,0.188139,0.020102,0.152581,0.152428
1,qwen2.5_0.5b_instruct_zero_shot_vllm,100,5.384482,0.224649,0.064147,0.175893,0.175831
2,qwen2.5_0.5b_instruct_lora,100,6.613557,0.259595,0.071622,0.195718,0.195709


In [38]:
import os

os.makedirs("outputs/metrics", exist_ok=True)
os.makedirs("outputs/predictions", exist_ok=True)

eval_df.to_csv(
    "outputs/predictions/qwen_lora_predictions.csv",
    index=False
)

final_comparison_df.to_csv(
    "outputs/metrics/final_metrics_comparison.csv",
    index=False
)

print("Saved:")
print("outputs/predictions/qwen_lora_predictions.csv")
print("outputs/metrics/final_metrics_comparison.csv")

Saved:
outputs/predictions/qwen_lora_predictions.csv
outputs/metrics/final_metrics_comparison.csv


In [39]:
from google.colab import files

files.download("outputs/metrics/final_metrics_comparison.csv")
files.download("outputs/predictions/qwen_lora_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>